# Bitget Cryptocurrency Data Scraper

Notebook này sử dụng Pydoll library để thu thập dữ liệu tiền điện tử từ Bitget exchange với các tính năng:
- ✅ Crawl đồng thời nhiều coin để tăng hiệu suất 
- ✅ Retry logic khi gặp lỗi
- ✅ Xuất dữ liệu ra Excel
- ✅ Real-time progress tracking

## Coins được crawl:
- Bitcoin
- Ethereum  
- Binance Coin
- Solana

## 1. Import Required Libraries

In [ ]:
import asyncio
import pandas as pd
from pydoll.browser.chrome import Chrome
from pydoll.constants import By
import time
from datetime import datetime

print("📦 Libraries imported successfully!")
print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Configuration

In [ ]:
# Configuration
list_coin = ['bitcoin', 'ethereum', 'binance', 'solana']
MAX_RETRIES = 2
WAIT_TIME = 3
ELEMENT_TIMEOUT = 10

print(f"🪙 Coins to scrape: {', '.join(list_coin)}")
print(f"🔄 Max retries: {MAX_RETRIES}")
print(f"⏱️  Wait time: {WAIT_TIME}s")
print(f"⏳ Element timeout: {ELEMENT_TIMEOUT}s")

## 3. Helper Functions

In [ ]:
async def get_element_text(page, element):
    """Helper function to get text from element using CDP commands"""
    try:
        if element and hasattr(element, '_object_id'):
            # Sử dụng Runtime.callFunctionOn để lấy text content
            command = {
                "id": 1,
                "method": "Runtime.callFunctionOn",
                "params": {
                    "functionDeclaration": "function() { return this.textContent || this.innerText || ''; }",
                    "objectId": element._object_id,
                    "returnByValue": True
                }
            }
            
            result = await page._connection_handler.execute_command(command)
            
            if result and 'result' in result and 'result' in result['result'] and 'value' in result['result']['result']:
                text = result['result']['result']['value'].strip()
                return text if text else "N/A"
            else:
                return "N/A"
        else:
            return "N/A"
    except Exception as e:
        print(f"⚠️ Error getting text from element: {e}")
        return "N/A"

print("✅ Helper functions defined")

## 4. Main Scraping Function

In [ ]:
async def fetch_coin_data(coin):
    """Fetch data for a single coin from Bitget"""
    url = f'https://www.bitget.com/price/{coin}'
    print(f"🔎 Starting crawl: {coin}")

    browser = Chrome()
    
    try:
        await browser.start()
        page = await browser.get_page()
        
        await page.go_to(url)
        print(f"📄 Page loaded: {coin}")
        
        await asyncio.sleep(WAIT_TIME)

        try:
            # Get price using XPath
            price_elem = await page.wait_element(
                By.XPATH, 
                '//span[@class="font-bold text-[40px] ltIpad:text-[32px] leading-[48px] ltIpad:leading-[38px] text-primaryText"]',
                timeout=ELEMENT_TIMEOUT
            )
            price = await get_element_text(page, price_elem) if price_elem else "N/A"

            # Get date
            try:
                date_elem = await page.wait_element(
                    By.XPATH, 
                    '//div[@class="text-[14px] mt-[24px] text-thirdText font-medium"]',
                    timeout=5
                )
                date = await get_element_text(page, date_elem) if date_elem else "N/A"
            except Exception:
                date = "N/A"

            data = {'coin': coin, 'price': price, 'date': date}
            
            try:
                # Get additional metrics
                labels = await page.find_elements(
                    By.XPATH, 
                    '//span[@class="text-[14px] text-[var(--content-secondary)]"]'
                )
                values = await page.find_elements(
                    By.XPATH, 
                    '//span[@class="text-[14px] font-[600]"]'
                )

                if labels and values:
                    keys = [await get_element_text(page, el) for el in labels[:5]]
                    vals = [await get_element_text(page, el) for el in values[:5]]
                    
                    limit = min(len(keys), len(vals))
                    for i in range(limit):
                        if keys[i] and vals[i] and keys[i] != "N/A" and vals[i] != "N/A":
                            clean_key = keys[i].strip(':').strip()
                            clean_val = vals[i].strip()
                            if clean_key and clean_val:
                                data[clean_key] = clean_val

            except Exception as e:
                print(f"⚠️ Unable to fetch additional info for {coin}: {e}")

            print(f"✅ Crawl completed: {coin}")
            return data

        except Exception as e:
            print(f"❌ Error finding element for {coin}: {e}")
            return {'coin': coin, 'error': str(e)}

    except Exception as e:
        print(f"❌ Error crawling {coin}: {e}")
        return {'coin': coin, 'error': str(e)}

    finally:
        try:
            await browser.stop()
        except Exception as e:
            print(f"⚠️ Error closing browser for {coin}: {e}")

print("✅ Main scraping function defined")

## 5. Retry Logic Function

In [ ]:
async def crawl_single_coin(coin):
    """Crawl a single coin with retry logic"""
    for attempt in range(MAX_RETRIES):
        try:
            result = await fetch_coin_data(coin)
            if 'error' not in result:
                return result
            else:
                print(f"🔄 Retrying {coin} attempt {attempt + 1}")
                await asyncio.sleep(2)
        except Exception as e:
            print(f"🔄 Error attempt {attempt + 1} for {coin}: {e}")
            if attempt < MAX_RETRIES - 1:
                await asyncio.sleep(3)
    
    return {'coin': coin, 'error': 'Failed after retries'}

print("✅ Retry logic function defined")

## 6. Data Export Function

In [ ]:
def export_data(processed_data):
    """Export data to Excel file"""
    try:
        df = pd.DataFrame(processed_data)
        excel_file = "bitget_coin_data.xlsx"
        df.to_excel(excel_file, index=False)
        print(f"📄 Data saved to {excel_file}")
        
        success_count = len([d for d in processed_data if 'error' not in d])
        error_count = len(processed_data) - success_count
        print(f"📊 Statistics: {success_count} successes, {error_count} errors")
        
        return df
        
    except Exception as e:
        print(f"❌ Error saving Excel file: {e}")
        return None

print("✅ Export function defined")

## 7. Main Execution Function

In [ ]:
async def main():
    """Main execution function with concurrent processing"""
    print("🚀 Starting cryptocurrency data crawling...")
    print(f"⏰ Start time: {datetime.now().strftime('%H:%M:%S')}")
    
    print(f"🧪 Testing with all coins concurrently...")
    
    try:
        # Create tasks for concurrent execution
        tasks = []
        for coin in list_coin:
            task = asyncio.create_task(crawl_single_coin(coin))
            tasks.append(task)
            print(f"📋 Created task for {coin}")
        
        print(f"🚀 Starting {len(tasks)} concurrent crawl tasks...")
        
        # Run all tasks concurrently
        all_data = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Process results and exceptions
        processed_data = []
        for i, result in enumerate(all_data):
            coin = list_coin[i]
            if isinstance(result, Exception):
                print(f"❌ Exception for {coin}: {result}")
                processed_data.append({'coin': coin, 'error': str(result)})
            else:
                processed_data.append(result)
        
        print(f"📋 All crawl tasks completed")
        print(f"⏰ End time: {datetime.now().strftime('%H:%M:%S')}")
        
        return processed_data
    
    except Exception as e:
        print(f"❌ Critical error in main: {e}")
        import traceback
        traceback.print_exc()
        return []

print("✅ Main execution function defined")

## 8. Run the Scraper

In [ ]:
# Run the main scraping process
start_time = time.time()

try:
    print("🎯 Starting scraper execution...")
    processed_data = await main()
    
    end_time = time.time()
    print(f"⏱️  Total execution time: {end_time - start_time:.2f} seconds")
    
    if processed_data:
        print(f"\n📋 Scraped {len(processed_data)} coins")
    else:
        print("❌ No data was scraped")
        
except Exception as e:
    print(f"❌ Critical error in execution: {e}")
    import traceback
    traceback.print_exc()

## 9. Export Data to Excel

In [ ]:
# Export data if we have results
if 'processed_data' in locals() and processed_data:
    df = export_data(processed_data)
    
    if df is not None:
        print("\n📊 Data Preview:")
        print(df.head())
        
        print("\n📈 Data Info:")
        print(df.info())
        
        # Show successful vs failed scrapes
        success_data = df[~df.columns.str.contains('error', case=False, na=False)]
        print(f"\n✅ Successfully scraped coins: {len(df[df.get('error', pd.Series()).isna()])}")
        print(f"❌ Failed scrapes: {len(df[df.get('error', pd.Series()).notna()])}")
else:
    print("❌ No data available to export")

## 10. Results Analysis

In [ ]:
# Analyze results if data exists
if 'df' in locals() and df is not None:
    print("🔍 DETAILED RESULTS ANALYSIS")
    print("=" * 50)
    
    for index, row in df.iterrows():
        coin = row.get('coin', 'Unknown')
        
        if 'error' in row and pd.notna(row['error']):
            print(f"❌ {coin.upper()}: FAILED - {row['error']}")
        else:
            print(f"✅ {coin.upper()}: SUCCESS")
            print(f"   💰 Price: {row.get('price', 'N/A')}")
            print(f"   📅 Date: {row.get('date', 'N/A')}")
            
            # Show additional metrics if available
            additional_cols = [col for col in row.index if col not in ['coin', 'price', 'date', 'error']]
            if additional_cols:
                print("   📊 Additional metrics:")
                for col in additional_cols[:3]:  # Show first 3 additional metrics
                    if pd.notna(row[col]):
                        print(f"      {col}: {row[col]}")
        print()
    
    print(f"📋 Summary: {len(df)} total coins processed")
    print(f"⏱️  Total time: {end_time - start_time:.2f} seconds")
    print(f"📄 Data saved to: bitget_coin_data.xlsx")
else:
    print("❌ No data available for analysis")